# Алгоритм обратного распространения ошибки

In [ ]:
from karpathy_code import *
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## Повторение

> В этом ноутбуке через **$\sigma$** мы будем обозначать произвольную функцию активации (не обязательно сигмоиду), но мыслить себе можно так, будто каждая функция активации - это обычная сигмоида

1. Что является результатом обработки вектора $\boldsymbol{x} \in \mathbb{X}$; $\boldsymbol{x} = (x_1, x_2, ..., x_n)$ искуственным нейроном?

    Это композиция нелинейного и линейного (афффиного) преобразований:

    $$\sigma(\boldsymbol{x}\boldsymbol{w} + b) = \sigma(w_1x_1 + w_2x_2 + ... + w_nx_n + b)$$

    Для удобства последующих обозначений результат 

2. Если мы объединим нейроны в слой, то что тогда будет являться выходом слоя номер $\mathcal{l}$, состоящего из $K$ нейронов?

    $$\sigma^{[\mathcal{l}]} = (\sigma_1^{[\mathcal{l}]}, \sigma_2^{[\mathcal{l}]}, ...,\sigma_k^{[\mathcal{l}]}, ... \sigma_K^{[\mathcal{l}]}),$$

    где каждый отдельный нейрон считается как $\sigma_k^{[\mathcal{l}]} = \sigma(\boldsymbol{x}\boldsymbol{w_k} + b_k)$, а результат всего слоя тогда можно записать в следующем виде:

    $$\sigma^{[\mathcal{l}]} = \sigma(W_\mathcal{l}\boldsymbol{x} + \boldsymbol{b}^{[\mathcal{l}]})$$

3. Как тогда можно представить всю нейронную сеть вместе?

    Всю нейронную сеть можно себе представлять как сложно устроенную функцию от гигантского количества параметров (десятки миллиардов на момент 2026 года). 
    
    $$a(\boldsymbol{x}; W) = \sigma^{[\mathcal{L}]}(\sigma^{[\mathcal{(L-1)}]}(...\sigma^{[\mathcal{1}]}(\boldsymbol{x})))$$
    
    
    Аппроксимация реальной зависимости происходит именно через параметризацию.


![alt](../data/NN.png)

Источник: К. В. Воронцов "Искусственные нейронные сети: градиентные методы оптимизации"

4. Как же теперь всё это дело обучать?

    Всё очень просто: давайте найдем градиент функции ошибки (лосса). 

    $$\nabla_W\mathbb{L} = \nabla_W\mathbb{L}(a(\boldsymbol{x}; W), y) 

## Вычисление градиента

Нейронные сети обучаются с помощью тех или иных модификаций градиентного спуска, а чтобы применять его, нужно уметь эффективно вычислять градиенты функции потерь по всем обучающим параметрам. Казалось бы, для какого-нибудь запутанного вычислительного графа это может быть очень сложной задачей, но на помощь спешит метод обратного распространения ошибки.

Начнем с более простых вещей, чтобы полноценно понять вычисление градиента такой сложной "перепараметризрованной функции"

Начнем с функции $f(x) = 3x^2 -4x + 5$

In [ ]:
def f(x):
    return 3*x**2 -4*x + 5

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-10, 10, 0.25)
ys = f(xs)
plt.plot(xs, ys)

In [ ]:
h = 0.1
x = 2/3
(f( x+ h) - f(x))/h

In [ ]:
# Немного усложним
a = 2.0
b = -3.0
c = 10.0
d = a*b + c
print(d)

In [ ]:
h = 0.0001

a = 2.0
b = -3.0
c = 10.0

d1 = a*b + c
a += h
d2 = a*b + c

print(f'{d1=}')
print(f'{d2=}')
print(f'slope={(d2-d1)/h}')

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        #self.grad = 0.0

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out


    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
e = a*b; e.label = 'e'
d = e + c; d.label = 'd'
d

In [ ]:
draw_dot(d)

In [ ]:
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a*b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'

In [ ]:
L.grad = 1.0 # dL\dL
f.grad = 4.0
d.grad = -2.0 # dL\dd = -2
c.grad = -2.0  # dL\dc = (dL\dd) * (dd\dc) = -2 * 1
e.grad = -2.0
b.grad = -4.0     # dL\db = dL\dd * dd\de * de\db = -2 * 2
a.grad = -6.0
draw_dot(L)


In [ ]:
def foo():
    h = 0.0001

    a = Value(2.0)
    b = Value(-3.0)
    c = Value(10.0)
    e = a*b; e.label = 'e'
    d = e + c; d.label = 'd'
    f = Value(-2.0, label='f')
    L = d * f; L.label = 'L'
    L1 = L.data

    a = Value(2.0)
    b = Value(-3.0)
    c = Value(10.0)
    e = a*b; e.label = 'e'
    d = e + c; d.label = 'd'
    f = Value(-2.0, label='f')
    L = d * f; L.label = 'L'
    L2 = L.data + h

    print((L2-L1)/h)

foo()

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self.grad = 0.0
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad = 1.0 * out.grad
            other.grad = 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad = other.data * out.grad
            other.grad = self.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self, ), label='tanh')

        def _backward():
            self.grad = (1 - t**2) * out.grad
        out._backward = _backward

        return out

In [ ]:
# Посмотрим, что будет внутри одного нейрона
# Входы
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# Веса
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# bias
b = Value(6.8813735870195432, label='b')

x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1*x2w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [ ]:
draw_dot(o)

In [ ]:
o.grad = 1.0
o._backward()

In [ ]:
n._backward()

In [ ]:
b._backward()

In [ ]:
x1w1x2w2._backward()

In [ ]:
x2w2._backward()
x1w1._backward()

Сделаем это рекурсивно для всех переменных  
Чтобы у нас всё работало как должно, нам надо вызывать `._backward()` только когда у родителей уже посчитан градиент, т.к. мы опираемся на .grad от родителя. 
Чтобы понять в каком порядке делать `._backward()` отсортируем все вершины, чтобы родители были в списке раньше их детей. Это можно сделать с помощью [топологической сортировки](https://ru.wikipedia.org/wiki/%D0%A2%D0%BE%D0%BF%D0%BE%D0%BB%D0%BE%D0%B3%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%81%D0%BE%D1%80%D1%82%D0%B8%D1%80%D0%BE%D0%B2%D0%BA%D0%B0).

In [ ]:
topo = []
visited = set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)
build_topo(o)
topo

In [ ]:
# Обнулим предварительно все градиенты в прошлой ячейке
o.grad = 1.0

topo: list[Value] = []
visited = set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)
build_topo(o)

for node in reversed(topo):
    node._backward()

## Честный подсчёт BackProp

Теперь формально опишем идею алгоритма.

![alt](../data/NN.png)

$$a^m(x_i) = \sigma_m^{[2]}(\sum_{h=0}^H w_{hm}u^h(x_i) + b^{[2]}_h)=\sigma_m^{[2]}(\sum_{h=0}^H w_{hm}u^h(x_i))$$

$$u^h = \sigma_h^{[1]}(\sum_{j=0}^J w_{jh}f_j(x_i))$$

БОО (только лишь для красивого примера) будем рассматривать функцию потерь следующего вида:

$$\mathbb{L_i} = \frac{1}{2}\sum_{m=1}^M(a^m(x_i) -y_i^m)^2$$

Решим промежуточную задачу: найдем частные производные

$$\frac{\partial \mathbb{L_i}}{\partial a^m} ; \ \ \frac{\partial \mathbb{L_i}}{\partial u^h}$$

Честно посчитаем:

$$\frac{\partial \mathbb{L_i}}{\partial a^m} = a^m(x_i)-y_i^m = \varepsilon_i^m$$

- это ошибка на выходном слое

$$\frac{\partial \mathbb{L_i}}{\partial u^h}=\sum_{m=1}^M \frac{\partial \mathbb{L_i}}{\partial a^m}(\sigma_m^{[2]}(\cdot))^\prime w_{hm} = \sum_{m=1}^M \varepsilon_i^m (\sigma_m^{[2]})^\prime w_{hm} = \varepsilon_i^h$$

- будем называть это *ошибкой на скрытом слоё*. Заметим, что $\varepsilon_i^h$ вычисляется через $\varepsilon_i^m$, если запустить сеть "задом-наперёд":

![alt](../data/b.png)


Теперь, зная частные ипроизводные

$$\frac{\partial \mathbb{L_i}}{\partial a^m} ; \ \ \frac{\partial \mathbb{L_i}}{\partial u^h}$$

легко выписать градиент $\mathbb{L}_i$ по весам $w$:

$$\frac{\partial \mathbb{L_i}}{\partial w_{hm}} = \frac{\partial \mathbb{L_i}}{\partial a^m} \frac{\partial a^m}{\partial w_{hm}} = \varepsilon_i^m (\sigma_m^{[2]})^\prime u^h(x_i), \ m=1..M, \ h=0..H$$

$$\frac{\partial \mathbb{L_i}}{\partial w_{jh}} = \frac{\partial \mathbb{L_i}}{\partial u^h} \frac{\partial u^h}{\partial w_{jh}} = \varepsilon_i^h (\sigma_h^{[1]})^\prime f_j(x_i), \ m=1..M, \ h=0..H$$


![alt](../data/backprop.gif)


## Затухающий градиент (Vanishing Gradient)


![alt](../data/vanishing_grad-1.webp)

Давайте внимательно посмотрим на то, как мы вычисляли градиенты для глубокой сети. Вспомните формулу для ошибки на скрытом слое:

$$
\varepsilon_i^h = \sum_{m=1}^M \varepsilon_i^m (\sigma_m^{[2]})' w_{hm}
$$

А если у нас не два слоя, а, скажем, 50? Тогда ошибка с выхода должна «дойти» до первого слоя, пройдя через 49 производных функций активации $(\sigma)'$ и 49 матриц весов.

В терминах нашего графа из `Value` это означает, что градиент (`.grad`) при движении от `L` к `a` последовательно умножается на:

1. Производную функции активации (например, для `tanh`: $1 - t^2$)
2. Веса (в нашем примере — `other.data` при умножении)
3. И так далее по цепочке.

### В чем проблема?

Большинство классических функций активации (сигмоида, $tanh$) имеют производные, которые на «насыщенных» участках (при больших по модулю значениях аргумента) стремятся к нулю.

- **Сигмоида:** $\sigma'(z) = \sigma(z)(1-\sigma(z)) \le 0.25$
- **$tanh$:** $tanh'(z) = 1 - tanh^2(z) \le 1$

Представьте, что 50 раз подряд мы умножаем число $\le 0.25$ (а часто и $\ll 0.25$). Градиент экспоненциально затухает.

**Следствия:**
- Слои, близкие к входу (первые слои), получают градиенты, близкие к нулю.
- Эти слои перестают обучаться (их веса не обновляются или обновляются крайне медленно).
- Сеть фактически обучает только последние слои, а первые остаются «случайными».

В нашем маленьком графе этого не видно, но если бы мы добавили 10 `tanh` подряд, то градиент на первом слое был бы произведением $(1-t^2)$ от каждого нейрона, что при $|t| > 0.9$ дает $\sim 0.19$, и после 10 слоев — $\sim 10^{-8}$.

> **Суть проблемы:** Произведение «маленьких» чисел (производных активаций) убивает градиент при движении от выхода к входу.

### Как эта проблема решалась в сетях ResNet?

Классические способы (выбор ReLU, аккуратная инициализация весов) помогали, но кардинально проблему решили **ResNet (Residual Networks)**, предложенные в 2015 году.

Идея ResNet проста и гениальна, и она напрямую следует из анализа формулы обратного распространения.

**Вместо того чтобы учить функцию $H(x)$ напрямую**, ResNet учит **«остаток»** (residual):

$$
H(x) = F(x) + x
$$

Где:
- $x$ — вход в блок (identity connection, «шорткат»)
- $F(x)$ — то, что выучивают два-три слоя (обычно с ReLU).

В терминах нашей лекции, выход слоя теперь зависит не только от преобразования $W\sigma(\cdot)$, но и от *неизменного* входа $x$.

#### Почему это спасает от затухающего градиента?

Рассмотрим прямое и обратное распространение через остаточный блок.

**Прямой проход:** 

$$
y = F(x) + x
$$

**Обратный проход:** мы хотим найти $\frac{\partial y}{\partial x}$.

$$
\frac{\partial y}{\partial x} = \frac{\partial}{\partial x}(F(x) + x) = \frac{\partial F(x)}{\partial x} + \mathbf{1}
$$

Посмотрите на это уравнение. Это ключевой момент.

При обратном распространении градиент $\frac{\partial L}{\partial x}$ (который идет дальше к предыдущим слоям) равен:

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \left( \frac{\partial F(x)}{\partial x} + \mathbf{1} \right) = \frac{\partial L}{\partial y} + \frac{\partial L}{\partial y} \cdot \frac{\partial F(x)}{\partial x}
$$

Теперь даже если $\frac{\partial F(x)}{\partial x}$ затухает (т.е. $\rightarrow 0$), градиент все равно получает **беспрепятственный вклад** от $\frac{\partial L}{\partial y}$ через слагаемое $+1$.

> **Образно:** В обычной сети градиент вынужден проходить сквозь «узкое горлышко» из производных активаций. В ResNet у градиента есть «путь обхода» (identity connection), где он не умножается на маленькие числа, а просто прибавляется к себе.

**На практике это означает:**
- Можно обучать сети с сотнями и тысячами слоев (например, ResNet-152, ResNet-1000).
- Градиент спокойно «протекает» от выхода ко входу.
- Ранние слои получают понятный сигнал для обновления, даже если $F(x)$ дает нулевой вклад.

![alt](../data/ResnetBlock.png)

ResNet показала, что если явно добавить входной сигнал к выходу блока, то градиентное затухание перестает быть фундаментальным ограничением. С этого момента глубокие сети стали действительно *глубокими*.